# DecisionTreeClassifier 하이퍼파라미터 상세 설명

sklearn의 sklearn.tree.DecisionTreeClassifier 주요 하이퍼파라미터를 정리해드립니다.

## 1. 트리 구조 제어 (과적합 방지 핵심)
## max_depth
- 역할: 트리의 최대 깊이 제한
- 기본값: None (완전히 순수해질 때까지, 또는 min_samples_split보다 샘플 수가 적을 때까지 확장)
- 사용법: max_depth=5 → 5단계까지만 분기
- 팁: 값이 너무 크면 과적합, 너무 작으면 과소적합. 보통 3~10 사이에서 튜닝 시작
## min_samples_split
- 역할: 노드를 분할하기 위해 필요한 최소 샘플 수
- 기본값: 2
- 사용법: min_samples_split=10 → 샘플이 10개 미만이면 더 이상 분할 안 함
- 타입: 정수(개수) 또는 실수(비율, 예: 0.05 = 전체의 5%)
## min_samples_leaf
- 역할: 리프 노드(끝 노드)가 가져야 할 최소 샘플 수
- 기본값: 1
- 사용법: min_samples_leaf=5 → 리프에 최소 5개 샘플 필요
- 팁: min_samples_split보다 과적합 방지에 더 효과적인 경우가 많음. 회귀나 불균형 ## 데이터에서 특히 유용
- max_leaf_nodes
- 역할: 리프 노드의 최대 개수 제한
- 기본값: None
- 사용법: max_leaf_nodes=20
- 특징: 지정하면 best-first 방식으로 트리 성장 (impurity 감소가 큰 노드부터 우선 분할)
## min_impurity_decrease
- 역할: 분할로 인한 불순도 감소량이 이 값 이상일 때만 분할 수행
- 기본값: 0.0
- 사용법: min_impurity_decrease=0.01
## 2. 분할 기준
## criterion
- 역할: 분할 품질을 측정하는 함수
- 옵션:
--  'gini' (기본값): 지니 불순도, 계산이 빠름
-- 'entropy': 정보 이득(information gain) 기반
-- 'log_loss': entropy와 동일한 방식
-- 실전: 대부분 결과 차이가 크지 않음. gini가 계산상 약간 더 빠름
## splitter
- 역할: 각 노드에서 분할을 선택하는 전략
- 옵션:
- 'best' (기본값): 최적의 분할 선택
- 'random': 무작위 분할 중 최선 선택 (RandomForest, ExtraTrees 계열에서 유용)
## 3. 피처 선택
## max_features
- 역할: 각 분할에서 고려할 최대 피처 수
- 옵션:
- None (기본값): 모든 피처 사용
- 'sqrt': √(전체 피처 수)
- 'log2': log2(전체 피처 수)
- 정수: 개수 지정
- 실수: 비율 지정
- 팁: 단일 트리에서는 보통 None 사용. 앙상블(랜덤포레스트)에서 다양성 확보 목적으로 주로 씀
## 4. 클래스 불균형 처리
- class_weight
- 역할: 클래스별 가중치 부여
- 옵션:
- None (기본값): 모든 클래스 동일 가중치
- 'balanced': 클래스 빈도에 반비례하게 자동 조정
- dict: 직접 지정, 예 {0: 1, 1: 5}
- 사용 시점: 불균형 데이터셋(사기 탐지, 이상 탐지 등)에서 필수적으로 고려
## 5. 기타
## random_state
- 역할: 재현성을 위한 난수 시드 (특히 splitter='random'이거나 동점 분할 처리 시 영향)
- 사용법: random_state=42
## ccp_alpha
- 역할: Cost-Complexity Pruning(비용 복잡도 가지치기) 강도
- 기본값: 0.0 (가지치기 없음)
- 사용법: 값이 클수록 더 많이 가지치기하여 단순한 트리 생성
- 팁: cost_complexity_pruning_path() 메서드로 최적 alpha 값을 미리 탐색 가능

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

# 기본 사용
clf = DecisionTreeClassifier(
    criterion='gini',
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    class_weight='balanced',
    random_state=42
)
clf.fit(X_train, y_train)

# GridSearch로 최적 하이퍼파라미터 탐색
param_grid = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 5, 10],
    'criterion': ['gini', 'entropy']
}

grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='f1_weighted',  # 불균형 데이터면 f1이나 roc_auc 추천
    n_jobs=-1
)
grid_search.fit(X_train, y_train)
print(grid_search.best_params_)